In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import cv2

In [3]:
import cv2
import torch
import sys

sys.path.insert(0, 'D:/Projects/Minor Project/Depth-Anything-V2/metric_depth')

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

from depth_anything_v2.dpt import DepthAnythingV2

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]}
}

encoder = 'vits' # or 'vits', 'vitb'
dataset = 'hypersim' # 'hypersim' for indoor model, 'vkitti' for outdoor model
max_depth = 10 # 20 for indoor model, 80 for outdoor model

check_path = f'D:/Projects/Minor Project/Parameters/DA-Parameter/depth_anything_v2_metric_hypersim_{encoder}.pth'

model = DepthAnythingV2(**model_configs[encoder], max_depth=max_depth)
model.load_state_dict(torch.load(check_path, map_location='cpu'))
model = model.to(DEVICE).eval()

In [4]:
print(DEVICE)

cuda


In [7]:
import torch.nn.functional as F

In [ ]:
# gets the depth features from the depth anything v2 encoder

def get_depth_encoder_features(depth_model, raw_img_np):
    features = {}
    input_size = {}

    def hook(module, input, output):
        features['out'] = output

    handle = depth_model.pretrained.blocks[-1].register_forward_hook(hook)

    with torch.no_grad():
        # Get the tensor to know exact H, W after preprocessing
        image, (h, w) = depth_model.image2tensor(raw_img_np, input_size=518)
        input_size['h'] = image.shape[2] // 14  # patch size is 14
        input_size['w'] = image.shape[3] // 14
        depth_model.forward(image.to(DEVICE))

    handle.remove()

    enc = features['out'][:, 1:, :]  # remove CLS token
    B, N, C = enc.shape
    
    pH = input_size['h']  # actual patch height
    pW = input_size['w']  # actual patch width
    
    print(f"Patch grid: {pH}x{pW}={pH*pW}, N={N}")  # should match

    spatial_map = enc.reshape(B, pH, pW, C).permute(0, 3, 1, 2)  # [1, 384, pH, pW]
    return spatial_map

In [8]:
# New projection to match SAM's [B, 256, 64, 64]
class DepthAnythingProjection(nn.Module):
    def __init__(self, in_channels=384, out_channels=256):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1),  # 384 → 256
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

    def forward(self, x):
        # x: [B, 384, 37, 37]
        x = self.proj(x)                                          # [B, 256, 37, 37]
        x = F.interpolate(x, size=(64, 64), mode='bilinear', align_corners=False)
        return x                                                   # [B, 256, 64, 64]

In [9]:
class PositionalEmbeddings(nn.Module):
    def __init__(self,num_tokens = 4096,embedding_dim = 256):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1,num_tokens,embedding_dim) * 0.02)

    def forward(self,x):
        return x + self.pos_embedding
    
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=256, nhead=8):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

    def forward(self, x):
        return self.transformer(x)

In [10]:
class DepthSAMFusion(nn.Module):
    def __init__(self):
        super().__init__()

        self.pos_encoder = PositionalEmbeddings()
        self.fusion_transformer = TransformerBlock()

        self.output_conv = nn.Conv2d(256,256,kernel_size=1)

    def forward(self,img_feature,depth_feature):
        # Features has shape [B,256,64,64]
        B,C,H,W = img_feature.shape

        # Flatten should start from dimension 2 and we get result in shape [B,256,4096]
        # then we reshape it to [B,4096,256]
        img_tokens = img_feature.flatten(2).permute(0,2,1)
        depth_tokens = depth_feature.flatten(2).permute(0,2,1)

        img_tokens = self.pos_encoder(img_tokens)
        depth_tokens = self.pos_encoder(depth_tokens)

        fused_tokens = self.fusion_transformer(img_tokens+depth_tokens)

        fused_grid = fused_tokens.permute(0,2,1).reshape(B,C,H,W)

        return self.output_conv(fused_grid)


In [11]:
class MonocularSAMModel(nn.Module):
    def __init__(self, sam_model, depth_anything_model, depth_proj, fusion_layer):
        super().__init__()
        self.sam          = sam_model
        self.depth_model  = depth_anything_model
        self.depth_proj   = depth_proj
        self.fusion_layer = fusion_layer

        # Freeze both encoders
        for param in self.sam.image_encoder.parameters():
            param.requires_grad = False
        for param in self.depth_model.pretrained.parameters():
            param.requires_grad = False

    def forward(self, image_input, raw_img_np):
        # 1. SAM image embeddings [B, 256, 64, 64]
        with torch.no_grad():
            image_embeddings = self.sam.image_encoder(image_input)

        # 2. Depth Anything encoder features [B, 384, 37, 37]
        depth_features = self.get_depth_features(raw_img_np)

        # 3. Project to SAM space [B, 256, 64, 64]
        depth_embeddings = self.depth_proj(depth_features)

        # 4. Fusion — your existing DepthSAMFusion works as-is
        fused_embeddings = self.fusion_layer(image_embeddings, depth_embeddings)

        # 5. SAM mask decoder
        sparse_embeddings, dense_embeddings = self.sam.prompt_encoder(
            points=None, boxes=None, masks=None,
        )
        low_res_masks, iou_predictions = self.sam.mask_decoder(
            image_embeddings=fused_embeddings,
            image_pe=self.sam.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=False,
        )
        return low_res_masks, iou_predictions

    def get_depth_features(self, raw_img_np):
        # Hook to extract ViT-S encoder output
        features = {}
        def hook(module, input, output):
            features['out'] = output
        handle = self.depth_model.pretrained.blocks[-1].register_forward_hook(hook)

        with torch.no_grad():
            self.depth_model.infer_image(raw_img_np)

        handle.remove()

        # [B, N+1, 384] → remove CLS → [B, 384, 37, 37]
        enc = features['out'][:, 1:, :]
        B, N, C = enc.shape
        H = W = int(N ** 0.5)
        return enc.reshape(B, H, W, C).permute(0, 3, 1, 2)